# Membangun Artefak XAI dan Prompt LLM untuk Rekomendasi AirSafe

Notebook ini digunakan untuk menghubungkan hasil forecasting PM2.5 dengan dua kebutuhan lanjutan, yaitu **visual explanation** dan **prompt package untuk LLM**.

Pada notebook sebelumnya, model LightGBM sudah dijelaskan menggunakan SHAP global importance. Notebook ini melanjutkan proses tersebut ke level yang lebih praktis, yaitu:

1. Membuat file prediksi per horizon yang dilengkapi top fitur SHAP.
2. Membuat waterfall plot SHAP untuk menjelaskan prediksi individual.
3. Membuat dependence plot untuk melihat hubungan fitur penting dengan kontribusi SHAP.
4. Menggabungkan prediksi H6, H12, dan H24.
5. Membentuk prompt package yang siap digunakan untuk LLM recommendation system.

Input utama notebook ini adalah dataset hasil ablation:

```text
outputs_ablation_datasets/dataset_h6_ablation.csv
outputs_ablation_datasets/dataset_h12_ablation.csv
outputs_ablation_datasets/dataset_h24_ablation.csv
```

Notebook juga membutuhkan best parameter LightGBM dari folder:

```text
outputs_phase2/
```

Selain itu, notebook membaca dua file prompt template:

```text
airsafe_recommendation_system.txt
airsafe_recommendation_user.txt
```

Dengan demikian, notebook ini berperan sebagai tahap transisi dari **model forecasting + SHAP explainability** menuju **LLM-based recommendation generation**.

## Alur XAI sampai Prompt Package

Alur kerja notebook ini dapat diringkas sebagai berikut:

```text
Dataset ablation H6 / H12 / H24
        ↓
Load best params LightGBM
        ↓
Latih ulang model pada fold terakhir
        ↓
Prediksi validation fold terakhir
        ↓
Hitung SHAP value per sample
        ↓
Simpan JSON prediksi + top SHAP
        ↓
Buat waterfall plot dan dependence plot
        ↓
Gabungkan prediksi H6, H12, dan H24
        ↓
Isi system prompt dan user prompt template
        ↓
Simpan prompt package untuk LLM
```

Notebook ini menggunakan **validation fold terakhir** sebagai sumber prediksi dan penjelasan. Fold terakhir dipilih karena merepresentasikan periode data paling baru dalam skema walk-forward validation.

Output notebook ini tidak hanya berupa grafik, tetapi juga file JSON dan CSV yang dapat digunakan untuk proses downstream, terutama untuk membuat rekomendasi kualitas udara berbasis LLM.

Secara konsep, notebook ini menjawab pertanyaan:

> Bagaimana prediksi PM2.5 dan faktor penyebabnya dapat dikemas menjadi input terstruktur untuk sistem rekomendasi berbasis LLM?

## Menyiapkan Konfigurasi XAI, Folder Output, dan Prompt Template

Cell ini menyiapkan library, konfigurasi utama, folder output, dan file prompt template yang akan digunakan untuk membuat prompt package.

Library yang digunakan adalah:

| Library | Fungsi |
|---|---|
| `json` | Membaca dan menyimpan file JSON |
| `pathlib.Path` | Mengatur path folder dan file |
| `numpy` | Operasi numerik |
| `pandas` | Membaca dan mengolah dataset |
| `matplotlib.pyplot` | Membuat dan menyimpan grafik |
| `shap` | Menghitung dan memvisualisasikan kontribusi fitur |
| `LGBMRegressor` | Melatih ulang model LightGBM |

Konfigurasi utama notebook adalah:

| Konfigurasi | Nilai | Fungsi |
|---|---:|---|
| `SEED` | 42 | Menjaga reproducibility |
| `N_FOLDS` | 4 | Jumlah fold walk-forward validation |
| `TOP_SHAP_PER_SAMPLE` | 6 | Jumlah fitur SHAP teratas per sample |
| `WATERFALL_ROWS_PER_STATION` | 5 | Jumlah waterfall plot per stasiun |
| `DEPENDENCE_TOPK_PER_STATION` | 2 | Jumlah fitur dependence plot teratas per stasiun |

Dataset yang digunakan berasal dari hasil ablation:

| Horizon | Dataset |
|---:|---|
| H6 | `outputs_ablation_datasets/dataset_h6_ablation.csv` |
| H12 | `outputs_ablation_datasets/dataset_h12_ablation.csv` |
| H24 | `outputs_ablation_datasets/dataset_h24_ablation.csv` |

Folder output utama notebook adalah:

```text
outputs_llm_xai/
```

Di dalam folder tersebut dibuat beberapa subfolder:

| Folder | Fungsi |
|---|---|
| `waterfall_plots/` | Menyimpan waterfall plot SHAP |
| `dependence_plots/` | Menyimpan dependence plot SHAP |
| `prediction_json/` | Menyimpan prediksi validation yang dilengkapi top SHAP |
| `prompt_packages/` | Menyimpan input prompt dan prompt package untuk LLM |

Cell ini juga membaca dua prompt template:

| File | Fungsi |
|---|---|
| `airsafe_recommendation_system.txt` | System prompt untuk mengatur peran dan aturan LLM |
| `airsafe_recommendation_user.txt` | User prompt template yang akan diisi dengan prediksi PM2.5 dan faktor SHAP |

Output cell menunjukkan bahwa kedua prompt template berhasil dibaca:

```text
System prompt loaded: airsafe_recommendation_system.txt
User prompt loaded  : airsafe_recommendation_user.txt
```

Artinya, notebook sudah siap membangun prompt package pada tahap akhir.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from lightgbm import LGBMRegressor

SEED = 42
N_FOLDS = 4
TOP_SHAP_PER_SAMPLE = 6
WATERFALL_ROWS_PER_STATION = 5
DEPENDENCE_TOPK_PER_STATION = 2

# dataset hasil ablation
DATASET_PATHS = {
    6: "outputs_ablation_datasets/dataset_h6_ablation.csv",
    12: "outputs_ablation_datasets/dataset_h12_ablation.csv",
    24: "outputs_ablation_datasets/dataset_h24_ablation.csv",
}

PHASE2_DIR = Path("outputs_phase2")
OUT_DIR = Path("outputs_llm_xai")
OUT_DIR.mkdir(exist_ok=True)

WATERFALL_DIR = OUT_DIR / "waterfall_plots"
DEPENDENCE_DIR = OUT_DIR / "dependence_plots"
PRED_JSON_DIR = OUT_DIR / "prediction_json"
PROMPT_DIR = OUT_DIR / "prompt_packages"

WATERFALL_DIR.mkdir(exist_ok=True)
DEPENDENCE_DIR.mkdir(exist_ok=True)
PRED_JSON_DIR.mkdir(exist_ok=True)
PROMPT_DIR.mkdir(exist_ok=True)

# prompt template dari file yang kamu upload
SYSTEM_PROMPT_PATH = "airsafe_recommendation_system.txt"
USER_PROMPT_PATH = "airsafe_recommendation_user.txt"

with open(SYSTEM_PROMPT_PATH, "r", encoding="utf-8") as f:
    SYSTEM_PROMPT_TEMPLATE = f.read()

with open(USER_PROMPT_PATH, "r", encoding="utf-8") as f:
    USER_PROMPT_TEMPLATE = f.read()

print("System prompt loaded:", SYSTEM_PROMPT_PATH)
print("User prompt loaded  :", USER_PROMPT_PATH)

System prompt loaded: airsafe_recommendation_system.txt
User prompt loaded  : airsafe_recommendation_user.txt


## Menyusun Fungsi Bantu untuk Prediksi, SHAP, dan JSON

Cell ini berisi kumpulan fungsi bantu yang menjadi fondasi utama notebook. Fungsi-fungsi ini digunakan untuk membaca dataset, menyiapkan fitur, membentuk fold validasi, menghitung SHAP, dan mengubah hasil prediksi ke format JSON.

### Fungsi `load_dataset()`

Fungsi ini membaca file dataset horizon menggunakan `pd.read_csv()` dengan `low_memory=False`, lalu mengubah kolom `datetime` dan `date` menjadi format datetime.

Dataset kemudian diurutkan berdasarkan:

```text
datetime
station_slug
```

Pengurutan ini penting karena data bersifat time series multi-stasiun.

### Fungsi `load_best_lgb_params()`

Fungsi ini membaca best params LightGBM dari folder:

```text
outputs_phase2/
```

File yang dibaca mengikuti pola:

```text
best_params_h6.json
best_params_h12.json
best_params_h24.json
```

Jika file tidak ditemukan, fungsi akan memberikan error. Hal ini wajar karena notebook ini memang bergantung pada hasil tuning model sebelumnya.

### Fungsi `prepare_xy_onehot()`

Fungsi ini memisahkan fitur `X` dan target `y`, lalu melakukan one-hot encoding untuk kolom object atau category.

Beberapa kolom dikeluarkan dari fitur:

| Kolom | Alasan Dikeluarkan |
|---|---|
| Target horizon | Label yang harus diprediksi |
| `datetime` | Kolom waktu mentah |
| `date` | Kolom tanggal mentah |
| `station_name` | Redundan dengan identitas stasiun |
| `lokasi` | Kolom teks lokasi |
| `season_simple` | Kolom string musim |
| `pm25_raw` | Backup PM2.5 asli |
| `pm25_clean_full` | Duplikasi PM2.5 final jika tersedia |

One-hot encoding diperlukan agar kolom kategorikal seperti `station_slug` dapat digunakan oleh LightGBM dan SHAP dalam bentuk numerik.

### Fungsi `make_walk_forward_folds()`

Fungsi ini membentuk 4-fold walk-forward validation. Data validation diambil dari bagian akhir timeline, sedangkan data training selalu berasal dari periode sebelum validation.

Fungsi ini juga menerapkan purge gap sesuai horizon:

| Horizon | Purge Gap |
|---:|---:|
| H6 | 6 jam |
| H12 | 12 jam |
| H24 | 24 jam |

Purge gap penting untuk mengurangi risiko leakage karena target dibuat dari nilai PM2.5 di masa depan.

### Fungsi `safe_python_value()` dan `sanitize_for_json()`

Dua fungsi ini digunakan untuk memastikan semua nilai dapat disimpan ke JSON.

Beberapa tipe data seperti `Timestamp`, `numpy integer`, `numpy float`, dan missing value perlu dikonversi terlebih dahulu agar tidak error saat disimpan menggunakan `json.dump()`.

### Fungsi `build_context_features()`

Fungsi ini memilih fitur-fitur penting dari satu baris data untuk dimasukkan ke bagian `context_features` pada JSON.

Fitur yang dipilih meliputi:

| Kelompok | Contoh Fitur |
|---|---|
| PM2.5 saat ini | `pm25` |
| Waktu | `hour_num`, `dayofweek`, `month` |
| Aktivitas | `is_rush_morning`, `is_rush_evening`, `is_workhour` |
| Cuaca | `temperature_2m`, `rain`, `wind_speed_10m`, `wind_u`, `wind_v` |
| Lag PM2.5 | `pm25_lag_1`, `pm25_lag_6`, `pm25_lag_24` |
| Rolling PM2.5 | `pm25_roll_mean_3`, `pm25_roll_mean_12`, `pm25_roll_mean_24` |
| Profil stasiun | `station_hour_mean_pm25`, `station_month_mean_pm25` |

Fitur konteks ini berguna agar output JSON tidak hanya berisi prediksi, tetapi juga informasi lingkungan yang membantu interpretasi.

### Fungsi `get_top_shap_items()`

Fungsi ini mengambil fitur SHAP paling penting dari satu sample.

Output yang dibuat adalah:

| Output | Makna |
|---|---|
| `top_abs` | Fitur dengan kontribusi SHAP terbesar secara absolut |
| `top_pos` | Fitur yang mendorong prediksi naik |
| `top_neg` | Fitur yang mendorong prediksi turun |

Notebook memakai `TOP_SHAP_PER_SAMPLE = 6`, sehingga setiap sample memiliki maksimal 6 fitur SHAP teratas.

### Fungsi `build_prediction_artifacts_for_horizon()`

Fungsi ini adalah fungsi inti untuk membangun artefak XAI per horizon.

Alurnya adalah:

1. Membaca dataset horizon.
2. Menentukan target, misalnya `target_pm25_t_plus_6`.
3. Membaca best params LightGBM.
4. Membentuk fold walk-forward.
5. Mengambil fold terakhir.
6. Melatih LightGBM pada training fold terakhir.
7. Melakukan prediksi pada validation fold terakhir.
8. Menghitung SHAP values untuk seluruh validation fold.
9. Membentuk list record berisi prediksi, error, context features, dan top SHAP features.

Output fungsi ini dipakai oleh cell berikutnya untuk menyimpan JSON, waterfall plot, dependence plot, dan prompt package.

In [2]:
def load_dataset(path):
    df = pd.read_csv(path, low_memory=False)

    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    sort_cols = [c for c in ["datetime", "station_slug"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)

    return df


def load_best_lgb_params(horizon):
    fp = PHASE2_DIR / f"best_params_h{horizon}.json"
    if not fp.exists():
        raise FileNotFoundError(f"Best params file tidak ditemukan: {fp}")

    with open(fp, "r") as f:
        obj = json.load(f)

    return obj.get("best_params", {})


def prepare_xy_onehot(df, target_col):
    exclude_cols = {
        target_col,
        "datetime",
        "date",
        "station_name",
        "lokasi",
        "season_simple",
        "pm25_raw",
        "pm25_clean_full",
    }
    exclude_cols = {c for c in exclude_cols if c in df.columns}

    X = df[[c for c in df.columns if c not in exclude_cols]].copy()
    y = df[target_col].copy()

    obj_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    X = pd.get_dummies(X, columns=obj_cols, dummy_na=False)

    return X, y


def make_walk_forward_folds(df, horizon, n_folds=4, valid_ratio_each=0.10):
    df = df.sort_values([c for c in ["datetime", "station_slug"] if c in df.columns]).reset_index(drop=True)
    unique_times = pd.Series(sorted(df["datetime"].dropna().unique()))

    n_times = len(unique_times)
    valid_size = max(1, int(n_times * valid_ratio_each))
    total_valid = n_folds * valid_size

    if total_valid >= n_times:
        raise ValueError("Terlalu banyak fold / valid_ratio terlalu besar.")

    first_valid_start_idx = n_times - total_valid
    folds = []

    for fold_id in range(n_folds):
        valid_start_idx = first_valid_start_idx + fold_id * valid_size
        valid_end_idx = min(valid_start_idx + valid_size - 1, n_times - 1)

        valid_start = pd.Timestamp(unique_times.iloc[valid_start_idx])
        valid_end = pd.Timestamp(unique_times.iloc[valid_end_idx])

        train_end = valid_start - pd.Timedelta(hours=horizon)

        train_mask = df["datetime"] < train_end
        valid_mask = (df["datetime"] >= valid_start) & (df["datetime"] <= valid_end)

        folds.append({
            "fold_id": fold_id + 1,
            "train_idx": df.index[train_mask].to_numpy(),
            "valid_idx": df.index[valid_mask].to_numpy(),
            "train_end": train_end,
            "valid_start": valid_start,
            "valid_end": valid_end,
        })

    return folds


def safe_python_value(v):
    if pd.isna(v):
        return None
    if isinstance(v, bool):
        return bool(v)
    if isinstance(v, np.bool_):
        return bool(v)
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return float(v)
    if isinstance(v, pd.Timestamp):
        return v.isoformat()
    return v


def sanitize_for_json(obj):
    if isinstance(obj, dict):
        return {str(k): sanitize_for_json(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [sanitize_for_json(v) for v in obj]
    else:
        return safe_python_value(obj)


def format_feature_name(feature_name):
    if feature_name.startswith("station_slug_"):
        slug = feature_name.replace("station_slug_", "")
        return f"station_slug={slug}"
    return feature_name


def build_context_features(row_dict):
    preferred = [
        "pm25",
        "hour_num", "dayofweek", "month",
        "is_weekend", "is_rush_morning", "is_rush_evening", "is_workhour",
        "season_dry_flag", "has_rain",
        "temperature_2m", "relative_humidity_2m", "rain",
        "surface_pressure", "wind_speed_10m", "wind_u", "wind_v",
        "pm25_lag_1", "pm25_lag_3", "pm25_lag_6", "pm25_lag_12", "pm25_lag_24",
        "pm25_roll_mean_3", "pm25_roll_mean_6", "pm25_roll_mean_12", "pm25_roll_mean_24",
        "station_hour_mean_pm25", "station_month_mean_pm25",
    ]
    context = {}
    for k in preferred:
        if k in row_dict:
            context[k] = safe_python_value(row_dict[k])
    return context


def get_top_shap_items(feature_names, feature_values, shap_values_row, top_k=6):
    items = []
    for f, val, sv in zip(feature_names, feature_values, shap_values_row):
        items.append({
            "feature": format_feature_name(f),
            "feature_value": safe_python_value(val),
            "shap_value": float(sv),
            "abs_shap_value": float(abs(sv)),
        })

    top_abs = sorted(items, key=lambda x: x["abs_shap_value"], reverse=True)[:top_k]
    top_pos = [x for x in items if x["shap_value"] > 0]
    top_pos = sorted(top_pos, key=lambda x: x["shap_value"], reverse=True)[:top_k]
    top_neg = [x for x in items if x["shap_value"] < 0]
    top_neg = sorted(top_neg, key=lambda x: x["shap_value"])[:top_k]

    def strip_abs(lst):
        return [
            {
                "feature": x["feature"],
                "feature_value": x["feature_value"],
                "shap_value": x["shap_value"],
            }
            for x in lst
        ]

    return strip_abs(top_abs), strip_abs(top_pos), strip_abs(top_neg)


def format_top_factors_text(top_abs_items, max_items=5):
    if not top_abs_items:
        return "-"
    feats = [x["feature"] for x in top_abs_items[:max_items]]
    return ", ".join(feats)


def build_prediction_artifacts_for_horizon(horizon, path):
    df_h = load_dataset(path)
    target_col = f"target_pm25_t_plus_{horizon}"
    best_params = load_best_lgb_params(horizon)

    folds = make_walk_forward_folds(df_h, horizon=horizon, n_folds=N_FOLDS, valid_ratio_each=0.10)
    last_fold = folds[-1]

    train_df = df_h.iloc[last_fold["train_idx"]].copy()
    valid_df = df_h.iloc[last_fold["valid_idx"]].copy()

    X_train, y_train = prepare_xy_onehot(train_df, target_col)
    X_valid, y_valid = prepare_xy_onehot(valid_df, target_col)

    X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)

    X_train = X_train.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    X_valid = X_valid.reset_index(drop=True)
    y_valid = y_valid.reset_index(drop=True)
    valid_meta = valid_df.copy().reset_index(drop=True)

    model = LGBMRegressor(
        random_state=SEED,
        verbose=-1,
        **best_params
    )
    model.fit(X_train, y_train)
    pred_valid = model.predict(X_valid)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_valid)

    valid_meta["prediction"] = pred_valid
    valid_meta["actual_target"] = y_valid
    valid_meta["absolute_error"] = (valid_meta["actual_target"] - valid_meta["prediction"]).abs()
    valid_meta["target_datetime"] = valid_meta["datetime"] + pd.Timedelta(hours=horizon)

    records = []
    for i in range(len(valid_meta)):
        row_meta = valid_meta.iloc[i].to_dict()
        row_x = X_valid.iloc[i]
        row_shap = shap_values[i]

        top_abs, top_pos, top_neg = get_top_shap_items(
            feature_names=X_valid.columns.tolist(),
            feature_values=row_x.values.tolist(),
            shap_values_row=row_shap,
            top_k=TOP_SHAP_PER_SAMPLE
        )

        rec = {
            "datetime": safe_python_value(row_meta.get("datetime")),
            "target_datetime": safe_python_value(row_meta.get("target_datetime")),
            "station_slug": safe_python_value(row_meta.get("station_slug")) if "station_slug" in valid_meta.columns else None,
            "station_id": safe_python_value(row_meta.get("station_id")) if "station_id" in valid_meta.columns else None,
            "horizon": horizon,
            "current_pm25": safe_python_value(row_meta.get("pm25")) if "pm25" in valid_meta.columns else None,
            "actual_target": safe_python_value(row_meta.get("actual_target")),
            "prediction": safe_python_value(row_meta.get("prediction")),
            "absolute_error": safe_python_value(row_meta.get("absolute_error")),
            "context_features": build_context_features(row_meta),
            "top_shap_features": top_abs,
            "top_positive_contributors": top_pos,
            "top_negative_contributors": top_neg,
            "top_factors_text": format_top_factors_text(top_abs),
        }
        records.append(rec)

    return {
        "horizon": horizon,
        "target_col": target_col,
        "last_fold": last_fold,
        "train_df": train_df,
        "valid_df": valid_df.reset_index(drop=True),
        "X_valid": X_valid,
        "model": model,
        "explainer": explainer,
        "shap_values": shap_values,
        "records": records,
    }

## Membangun Prediction JSON, Waterfall Plot, dan Dependence Plot

Cell ini menjalankan proses utama untuk membuat artefak XAI pada tiga horizon prediksi: H6, H12, dan H24.

Untuk setiap horizon, notebook menjalankan fungsi:

```text
build_prediction_artifacts_for_horizon()
```

Fungsi tersebut menghasilkan model, prediksi validation, SHAP values, dan metadata yang kemudian disimpan dalam beberapa bentuk output.

### Prediction JSON per Horizon

Untuk setiap horizon, notebook menyimpan file JSON yang berisi prediksi validation fold terakhir beserta top fitur SHAP.

File yang dihasilkan adalah:

| Horizon | File JSON | File CSV Ringkas |
|---:|---|---|
| H6 | `outputs_llm_xai/prediction_json/prediction_h6_with_top_shap.json` | `outputs_llm_xai/prediction_json/prediction_h6_with_top_shap.csv` |
| H12 | `outputs_llm_xai/prediction_json/prediction_h12_with_top_shap.json` | `outputs_llm_xai/prediction_json/prediction_h12_with_top_shap.csv` |
| H24 | `outputs_llm_xai/prediction_json/prediction_h24_with_top_shap.json` | `outputs_llm_xai/prediction_json/prediction_h24_with_top_shap.csv` |

Setiap record prediksi berisi:

| Komponen | Makna |
|---|---|
| `datetime` | Waktu input prediksi |
| `target_datetime` | Waktu target sesuai horizon |
| `station_slug` | Identitas stasiun |
| `horizon` | Horizon prediksi |
| `current_pm25` | PM2.5 pada waktu input |
| `actual_target` | Nilai PM2.5 aktual pada target horizon |
| `prediction` | Hasil prediksi model |
| `absolute_error` | Selisih absolut antara aktual dan prediksi |
| `context_features` | Fitur konteks penting |
| `top_shap_features` | Fitur SHAP paling berpengaruh |
| `top_positive_contributors` | Fitur yang menaikkan prediksi |
| `top_negative_contributors` | Fitur yang menurunkan prediksi |
| `top_factors_text` | Ringkasan nama fitur penting untuk prompt LLM |

### Waterfall Plot SHAP

Notebook membuat waterfall plot untuk beberapa baris awal per stasiun.

Konfigurasi yang digunakan adalah:

```text
WATERFALL_ROWS_PER_STATION = 5
```

Artinya, untuk setiap horizon dan setiap stasiun, notebook mengambil 5 baris awal dari validation fold terakhir untuk divisualisasikan dalam waterfall plot.

Waterfall plot menjelaskan bagaimana fitur-fitur tertentu mendorong prediksi naik atau turun dari baseline value model menuju nilai prediksi akhir.

Output waterfall plot disimpan di folder:

```text
outputs_llm_xai/waterfall_plots/
```

Notebook juga menyimpan daftar file waterfall ke:

```text
outputs_llm_xai/waterfall_manifest.csv
```

### Dependence Plot SHAP

Notebook juga membuat dependence plot per stasiun.

Untuk setiap stasiun, notebook memilih 2 fitur teratas berdasarkan rata-rata absolute SHAP pada validation fold.

Konfigurasi yang digunakan adalah:

```text
DEPENDENCE_TOPK_PER_STATION = 2
```

Dependence plot digunakan untuk melihat hubungan antara nilai fitur dan kontribusi SHAP. Grafik ini membantu membaca apakah nilai fitur yang lebih besar cenderung menaikkan atau menurunkan prediksi.

Output dependence plot disimpan di folder:

```text
outputs_llm_xai/dependence_plots/
```

Daftar file dependence plot disimpan ke:

```text
outputs_llm_xai/dependence_manifest.csv
```

### Output Gabungan Semua Horizon

Setelah H6, H12, dan H24 selesai diproses, notebook menyimpan JSON gabungan:

```text
outputs_llm_xai/prediction_json/prediction_all_horizons_with_top_shap.json
```

Output terminal menunjukkan bahwa file utama berhasil disimpan:

```text
Saved:
- outputs_llm_xai/prediction_json/prediction_all_horizons_with_top_shap.json
- outputs_llm_xai/waterfall_manifest.csv
- outputs_llm_xai/dependence_manifest.csv
```

In [3]:
all_prediction_json = {}
waterfall_manifest = []
dependence_manifest = []

for horizon, path in DATASET_PATHS.items():
    print(f"\n===== BUILD XAI ARTIFACTS H{horizon} =====")
    art = build_prediction_artifacts_for_horizon(horizon, path)

    # =========================
    # save prediction JSON per horizon
    # =========================
    horizon_json = {
        "horizon": horizon,
        "target_column": art["target_col"],
        "validation_window": {
            "fold_id": int(art["last_fold"]["fold_id"]),
            "train_end": safe_python_value(art["last_fold"]["train_end"]),
            "valid_start": safe_python_value(art["last_fold"]["valid_start"]),
            "valid_end": safe_python_value(art["last_fold"]["valid_end"]),
        },
        "predictions": art["records"],
    }

    json_path = PRED_JSON_DIR / f"prediction_h{horizon}_with_top_shap.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(sanitize_for_json(horizon_json), f, ensure_ascii=False, indent=2)

    # csv ringkas juga
    pd.DataFrame(art["records"]).to_csv(
        PRED_JSON_DIR / f"prediction_h{horizon}_with_top_shap.csv",
        index=False
    )

    print("Saved prediction JSON:", json_path)
    all_prediction_json[f"h{horizon}"] = horizon_json

    # =========================
    # waterfall plot: 5 rows per station
    # =========================
    valid_meta = art["valid_df"].copy()
    valid_meta["target_datetime"] = valid_meta["datetime"] + pd.Timedelta(hours=horizon)

    selected_rows = (
        valid_meta.sort_values(["station_slug", "datetime"])
        .groupby("station_slug", group_keys=False)
        .head(WATERFALL_ROWS_PER_STATION)
        .reset_index()
    )

    for _, row in selected_rows.iterrows():
        idx = int(row["index"])
        station = row["station_slug"]

        explanation = shap.Explanation(
            values=art["shap_values"][idx],
            base_values=art["explainer"].expected_value,
            data=art["X_valid"].iloc[idx].values,
            feature_names=art["X_valid"].columns.tolist()
        )

        plt.figure(figsize=(10, 6))
        shap.plots.waterfall(explanation, max_display=12, show=False)
        plt.title(
            f"Waterfall SHAP - H{horizon} - {station}\n"
            f"{pd.to_datetime(row['datetime'])} -> {pd.to_datetime(row['target_datetime'])}"
        )
        plt.tight_layout()

        fname = f"waterfall_h{horizon}_{station}_{pd.to_datetime(row['datetime']).strftime('%Y%m%d_%H%M%S')}.png"
        fpath = WATERFALL_DIR / fname
        plt.savefig(fpath, dpi=200, bbox_inches="tight")
        plt.close()

        waterfall_manifest.append({
            "horizon": horizon,
            "station_slug": station,
            "datetime": safe_python_value(pd.to_datetime(row["datetime"])),
            "target_datetime": safe_python_value(pd.to_datetime(row["target_datetime"])),
            "file_path": str(fpath),
        })

    # =========================
    # dependence plot per station
    # pakai seluruh row valid station tsb, top-2 feature by mean abs shap
    # =========================
    for station in sorted(valid_meta["station_slug"].dropna().unique()):
        station_mask = valid_meta["station_slug"].eq(station).values
        station_X = art["X_valid"].loc[station_mask].copy()
        station_shap = art["shap_values"][station_mask]

        if len(station_X) < 10:
            continue

        mean_abs_station = np.abs(station_shap).mean(axis=0)
        top_feature_idx = np.argsort(mean_abs_station)[::-1][:DEPENDENCE_TOPK_PER_STATION]
        top_features = [station_X.columns[i] for i in top_feature_idx]

        for feat in top_features:
            plt.figure(figsize=(8, 6))
            shap.dependence_plot(
                feat,
                station_shap,
                station_X,
                show=False,
                interaction_index=None
            )
            plt.title(f"Dependence SHAP - H{horizon} - {station} - {feat}")
            plt.tight_layout()

            fname = f"dependence_h{horizon}_{station}_{feat}.png".replace("/", "_")
            fpath = DEPENDENCE_DIR / fname
            plt.savefig(fpath, dpi=200, bbox_inches="tight")
            plt.close()

            dependence_manifest.append({
                "horizon": horizon,
                "station_slug": station,
                "feature": feat,
                "file_path": str(fpath),
            })

# simpan gabungan semua horizon
with open(PRED_JSON_DIR / "prediction_all_horizons_with_top_shap.json", "w", encoding="utf-8") as f:
    json.dump(sanitize_for_json(all_prediction_json), f, ensure_ascii=False, indent=2)

pd.DataFrame(waterfall_manifest).to_csv(OUT_DIR / "waterfall_manifest.csv", index=False)
pd.DataFrame(dependence_manifest).to_csv(OUT_DIR / "dependence_manifest.csv", index=False)

print("\nSaved:")
print("-", PRED_JSON_DIR / "prediction_all_horizons_with_top_shap.json")
print("-", OUT_DIR / "waterfall_manifest.csv")
print("-", OUT_DIR / "dependence_manifest.csv")


===== BUILD XAI ARTIFACTS H6 =====
Saved prediction JSON: outputs_llm_xai/prediction_json/prediction_h6_with_top_shap.json

===== BUILD XAI ARTIFACTS H12 =====
Saved prediction JSON: outputs_llm_xai/prediction_json/prediction_h12_with_top_shap.json


/home/nafisnaufal1426/adit/envAdit/envs/semuanya/lib/python3.10/site-packages/shap/plots/_scatter.py:641: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=figsize)



===== BUILD XAI ARTIFACTS H24 =====
Saved prediction JSON: outputs_llm_xai/prediction_json/prediction_h24_with_top_shap.json


/tmp/ipykernel_278491/1882885018.py:61: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(10, 6))
/tmp/ipykernel_278491/1882885018.py:61: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(10, 6))
/tmp/ipykernel_278491/1882885018.py:61: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much mem


Saved:
- outputs_llm_xai/prediction_json/prediction_all_horizons_with_top_shap.json
- outputs_llm_xai/waterfall_manifest.csv
- outputs_llm_xai/dependence_manifest.csv


...

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

In [4]:
# =========================================================
# PATCH: Regenerate Waterfall Plot H12
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

horizon = 12

print(f"===== REGENERATE WATERFALL H{horizon} =====")

# Build ulang artefak H12
art_h12 = build_prediction_artifacts_for_horizon(
    horizon=horizon,
    path=DATASET_PATHS[horizon]
)

valid_meta = art_h12["valid_df"].copy().reset_index(drop=True)
valid_meta["valid_pos"] = np.arange(len(valid_meta))
valid_meta["target_datetime"] = valid_meta["datetime"] + pd.Timedelta(hours=horizon)

print("\nJumlah row validation per stasiun:")
display(valid_meta["station_slug"].value_counts())

# Ambil 5 row pertama per stasiun secara eksplisit
selected_rows = (
    valid_meta
    .sort_values(["station_slug", "datetime"])
    .groupby("station_slug", group_keys=False)
    .head(WATERFALL_ROWS_PER_STATION)
    .reset_index(drop=True)
)

print("\nJumlah waterfall yang akan dibuat per stasiun:")
display(selected_rows.groupby("station_slug").size())

print("\nTotal waterfall H12 yang akan dibuat:", len(selected_rows))

# Hapus file waterfall H12 lama supaya tidak tercampur
for old_fp in WATERFALL_DIR.glob("waterfall_h12_*.png"):
    old_fp.unlink()

new_waterfall_manifest = []

plt.close("all")

for _, row in selected_rows.iterrows():
    valid_pos = int(row["valid_pos"])
    station = str(row["station_slug"])
    dt = pd.to_datetime(row["datetime"])
    target_dt = pd.to_datetime(row["target_datetime"])

    explanation = shap.Explanation(
        values=art_h12["shap_values"][valid_pos],
        base_values=art_h12["explainer"].expected_value,
        data=art_h12["X_valid"].iloc[valid_pos].values,
        feature_names=art_h12["X_valid"].columns.tolist()
    )

    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(explanation, max_display=12, show=False)

    plt.title(
        f"Waterfall SHAP - H{horizon} - {station}\n"
        f"{dt} -> {target_dt}"
    )
    plt.tight_layout()

    fname = (
        f"waterfall_h{horizon}_"
        f"{station}_"
        f"pos{valid_pos}_"
        f"{dt.strftime('%Y%m%d_%H%M%S')}.png"
    )
    fpath = WATERFALL_DIR / fname

    plt.savefig(fpath, dpi=200, bbox_inches="tight")
    plt.close("all")

    new_waterfall_manifest.append({
        "horizon": horizon,
        "station_slug": station,
        "datetime": dt.isoformat(),
        "target_datetime": target_dt.isoformat(),
        "valid_pos": valid_pos,
        "file_path": str(fpath),
    })

# Update waterfall_manifest.csv khusus H12
manifest_path = OUT_DIR / "waterfall_manifest.csv"

if manifest_path.exists():
    old_manifest = pd.read_csv(manifest_path)
    old_manifest = old_manifest[old_manifest["horizon"] != horizon].copy()
else:
    old_manifest = pd.DataFrame()

new_manifest = pd.DataFrame(new_waterfall_manifest)

final_manifest = pd.concat([old_manifest, new_manifest], ignore_index=True)
final_manifest.to_csv(manifest_path, index=False)

print("\nSelesai regenerate waterfall H12.")
print("Manifest updated:", manifest_path)
print("Jumlah file H12 baru:", len(new_manifest))

display(new_manifest.head())

===== REGENERATE WATERFALL H12 =====

Jumlah row validation per stasiun:


station_slug
dki1-bundaran-hi      3116
dki2-kelapa-gading    3116
dki3-jagakarsa        3116
dki4-lubang-buaya     3116
dki5-kebun-jeruk      3116
Name: count, dtype: int64


Jumlah waterfall yang akan dibuat per stasiun:


station_slug
dki1-bundaran-hi      5
dki2-kelapa-gading    5
dki3-jagakarsa        5
dki4-lubang-buaya     5
dki5-kebun-jeruk      5
dtype: int64


Total waterfall H12 yang akan dibuat: 25

Selesai regenerate waterfall H12.
Manifest updated: outputs_llm_xai/waterfall_manifest.csv
Jumlah file H12 baru: 25


,horizon,station_slug,datetime,target_datetime,valid_pos,file_path
0,12,dki1-bundaran-hi,2025-12-12T16:00:00,2025-12-13T04:00:00,0,outputs_llm_xai/waterfall_plots/waterfall_h12_...
1,12,dki1-bundaran-hi,2025-12-12T17:00:00,2025-12-13T05:00:00,5,outputs_llm_xai/waterfall_plots/waterfall_h12_...
2,12,dki1-bundaran-hi,2025-12-12T18:00:00,2025-12-13T06:00:00,10,outputs_llm_xai/waterfall_plots/waterfall_h12_...
3,12,dki1-bundaran-hi,2025-12-12T19:00:00,2025-12-13T07:00:00,15,outputs_llm_xai/waterfall_plots/waterfall_h12_...
4,12,dki1-bundaran-hi,2025-12-12T20:00:00,2025-12-13T08:00:00,20,outputs_llm_xai/waterfall_plots/waterfall_h12_...


## Mengubah Prediksi Multi-Horizon Menjadi Prompt Package LLM

Cell ini menggabungkan hasil prediksi H6, H12, dan H24, lalu mengubahnya menjadi input prompt untuk sistem rekomendasi berbasis LLM.

### Metadata Stasiun atau Sekolah

Pada awal cell terdapat konfigurasi:

```text
STATION_META_PATH = None
```

Artinya, saat ini notebook belum menggunakan file metadata sekolah.

Jika nanti tersedia file mapping stasiun ke sekolah, file tersebut dapat diisi dengan kolom minimal:

| Kolom | Fungsi |
|---|---|
| `station_slug` | Menghubungkan sekolah dengan stasiun pemantauan |
| `school_id` | ID sekolah |
| `school_name` | Nama sekolah |
| `district` | Wilayah sekolah |
| `risk_level` | Level risiko operasional |

Karena `STATION_META_PATH` masih `None`, notebook menggunakan fallback:

| Kolom | Nilai Fallback |
|---|---|
| `school_id` | Diisi dengan `station_slug` |
| `school_name` | Diisi dengan `station_slug` |
| `district` | Diisi `None` |
| `risk_level` | Diisi `TODO_SET_RISK_LEVEL` |

Artinya, prompt package sudah bisa dibuat, tetapi metadata sekolah dan risk level masih perlu dilengkapi jika sistem ingin digunakan dalam skenario sekolah nyata.

### Menggabungkan Prediksi H6, H12, dan H24

Fungsi `records_to_df()` mengubah list prediksi JSON menjadi DataFrame ringkas.

Setiap horizon menghasilkan kolom:

| Horizon | Kolom Prediksi | Kolom Faktor |
|---:|---|---|
| H6 | `pm25_6h` | `top_factors_6h` |
| H12 | `pm25_12h` | `top_factors_12h` |
| H24 | `pm25_24h` | `top_factors_24h` |

Ketiga DataFrame kemudian digabung berdasarkan:

```text
datetime
station_slug
```

Hasil akhirnya berisi prediksi PM2.5 untuk 6 jam, 12 jam, dan 24 jam pada waktu dan stasiun yang sama.

### Menggabungkan Faktor Penting

Fungsi `combine_top_factors()` mengambil dua faktor teratas dari setiap horizon, lalu menggabungkannya menjadi satu daftar unik.

Contohnya, jika faktor penting dari H6, H12, dan H24 banyak mengandung `pm25`, `pm25_lag_1`, dan `pm25_roll_mean_48`, maka kolom `top_factors` akan berisi gabungan faktor tersebut tanpa duplikasi.

Kolom `top_factors` inilah yang akan dimasukkan ke user prompt.

### Membentuk Prompt Package

Untuk setiap baris data, notebook mengisi `USER_PROMPT_TEMPLATE` menggunakan beberapa variabel:

| Placeholder | Isi |
|---|---|
| `school_name` | Nama sekolah atau fallback station_slug |
| `district` | Wilayah sekolah jika tersedia |
| `pm25_6h` | Prediksi PM2.5 6 jam |
| `pm25_12h` | Prediksi PM2.5 12 jam |
| `pm25_24h` | Prediksi PM2.5 24 jam |
| `risk_level` | Level risiko |
| `top_factors` | Faktor utama dari SHAP |

Setiap prompt package berisi:

| Kolom | Makna |
|---|---|
| `school_id` | ID sekolah atau station_slug |
| `school_name` | Nama sekolah atau station_slug |
| `district` | Wilayah sekolah |
| `station_slug` | Identitas stasiun |
| `datetime` | Waktu prediksi |
| `risk_level` | Level risiko operasional |
| `pm25_6h` | Prediksi horizon 6 jam |
| `pm25_12h` | Prediksi horizon 12 jam |
| `pm25_24h` | Prediksi horizon 24 jam |
| `top_factors` | Faktor penting dari SHAP |
| `system_prompt` | Prompt sistem untuk LLM |
| `user_prompt` | Prompt user yang sudah diisi data prediksi |

### Output yang Disimpan

Cell ini menyimpan tiga file utama:

```text
outputs_llm_xai/prompt_packages/llm_prompt_input.csv
outputs_llm_xai/prompt_packages/llm_prompt_packages.jsonl
outputs_llm_xai/prompt_packages/llm_prompt_packages.json
```

File `llm_prompt_input.csv` berisi data tabular yang menjadi bahan prompt.

File `llm_prompt_packages.jsonl` cocok digunakan untuk pipeline batch inference karena setiap baris adalah satu record JSON.

File `llm_prompt_packages.json` cocok digunakan untuk inspeksi manual atau aplikasi yang membutuhkan satu file JSON berisi list.

Output terminal juga menampilkan contoh satu prompt package. Dari contoh tersebut terlihat bahwa prompt sudah berisi prediksi PM2.5 H6, H12, H24, top factors, system prompt, dan user prompt.

In [5]:
# OPTIONAL:
# kalau nanti kamu punya mapping station -> school, bisa isi file ini
# kolom minimal yang didukung:
# station_slug, school_id, school_name, district, risk_level
STATION_META_PATH = None   # contoh: "station_school_meta.csv"

if STATION_META_PATH is not None:
    station_meta = pd.read_csv(STATION_META_PATH)
else:
    station_meta = None

def records_to_df(records, horizon):
    rows = []
    for r in records:
        rows.append({
            "datetime": r["datetime"],
            "station_slug": r["station_slug"],
            f"pm25_{horizon}h": r["prediction"],
            f"top_factors_{horizon}h": r["top_factors_text"],
        })
    df = pd.DataFrame(rows)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    return df

# load dari all_prediction_json yang baru dibuat
df6 = records_to_df(all_prediction_json["h6"]["predictions"], 6)
df12 = records_to_df(all_prediction_json["h12"]["predictions"], 12)
df24 = records_to_df(all_prediction_json["h24"]["predictions"], 24)

merged = df6.merge(df12, on=["datetime", "station_slug"], how="inner")
merged = merged.merge(df24, on=["datetime", "station_slug"], how="inner")

# merge metadata kalau ada
if station_meta is not None and "station_slug" in station_meta.columns:
    merged = merged.merge(station_meta, on="station_slug", how="left")
else:
    # fallback proxy pakai station sebagai entity
    merged["school_id"] = merged["station_slug"]
    merged["school_name"] = merged["station_slug"]
    merged["district"] = None
    merged["risk_level"] = "TODO_SET_RISK_LEVEL"

def combine_top_factors(row):
    factors = []
    for c in ["top_factors_6h", "top_factors_12h", "top_factors_24h"]:
        val = row.get(c, None)
        if pd.notna(val) and str(val).strip() not in {"", "-"}:
            parts = [x.strip() for x in str(val).split(",")]
            factors.extend(parts[:2])  # ambil 2 teratas per horizon
    # unique, preserve order
    seen = set()
    uniq = []
    for f in factors:
        if f not in seen:
            seen.add(f)
            uniq.append(f)
    return ", ".join(uniq[:6]) if uniq else "-"

merged["top_factors"] = merged.apply(combine_top_factors, axis=1)

# save input siap prompt
merged.to_csv(PROMPT_DIR / "llm_prompt_input.csv", index=False)

prompt_records = []
for _, row in merged.iterrows():
    user_prompt_filled = USER_PROMPT_TEMPLATE.format(
        school_name=row.get("school_name"),
        district=row.get("district"),
        pm25_6h=row.get("pm25_6h"),
        pm25_12h=row.get("pm25_12h"),
        pm25_24h=row.get("pm25_24h"),
        risk_level=row.get("risk_level"),
        top_factors=row.get("top_factors"),
    )

    prompt_records.append({
        "school_id": safe_python_value(row.get("school_id")),
        "school_name": safe_python_value(row.get("school_name")),
        "district": safe_python_value(row.get("district")),
        "station_slug": safe_python_value(row.get("station_slug")),
        "datetime": safe_python_value(row.get("datetime")),
        "risk_level": safe_python_value(row.get("risk_level")),
        "pm25_6h": safe_python_value(row.get("pm25_6h")),
        "pm25_12h": safe_python_value(row.get("pm25_12h")),
        "pm25_24h": safe_python_value(row.get("pm25_24h")),
        "top_factors": safe_python_value(row.get("top_factors")),
        "system_prompt": SYSTEM_PROMPT_TEMPLATE,
        "user_prompt": user_prompt_filled,
    })

# save jsonl untuk LLM batch/pipeline
jsonl_path = PROMPT_DIR / "llm_prompt_packages.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for rec in prompt_records:
        f.write(json.dumps(sanitize_for_json(rec), ensure_ascii=False) + "\n")

# save json biasa juga
with open(PROMPT_DIR / "llm_prompt_packages.json", "w", encoding="utf-8") as f:
    json.dump(sanitize_for_json(prompt_records), f, ensure_ascii=False, indent=2)

print("Saved:")
print("-", PROMPT_DIR / "llm_prompt_input.csv")
print("-", jsonl_path)
print("-", PROMPT_DIR / "llm_prompt_packages.json")
print("\nContoh 1 prompt package:")
print(json.dumps(sanitize_for_json(prompt_records[0]), ensure_ascii=False, indent=2)[:2000])

Saved:
- outputs_llm_xai/prompt_packages/llm_prompt_input.csv
- outputs_llm_xai/prompt_packages/llm_prompt_packages.jsonl
- outputs_llm_xai/prompt_packages/llm_prompt_packages.json

Contoh 1 prompt package:
{
  "school_id": "dki1-bundaran-hi",
  "school_name": "dki1-bundaran-hi",
  "district": null,
  "station_slug": "dki1-bundaran-hi",
  "datetime": "2025-12-12T21:00:00",
  "risk_level": "TODO_SET_RISK_LEVEL",
  "pm25_6h": 55.006459562443105,
  "pm25_12h": 56.00324813508579,
  "pm25_24h": 56.81110725625537,
  "top_factors": "pm25, pm25_lag_1, pm25_lag_24, pm25_roll_mean_48",
  "system_prompt": "You are AirSafe School, an AI assistant that writes operational air-quality recommendations for schools in Jakarta.\n\nYour task is to convert PM2.5 forecasts, risk level, and model explanation factors into clear Bahasa Indonesia recommendations for school principals.\n\nRules:\n- Write in Bahasa Indonesia.\n- Be specific, practical, and calm.\n- Do not invent data not provided in the input.\n-

# Peran Notebook dalam Sistem AirSafe

Notebook ini menjadi tahap penting untuk mengubah hasil model forecasting menjadi artefak yang lebih siap digunakan oleh sistem rekomendasi.

Secara ringkas, notebook ini menghasilkan tiga jenis output:

| Jenis Output | Fungsi |
|---|---|
| Prediction JSON | Menyimpan prediksi PM2.5 dan faktor SHAP per sample |
| XAI Plot | Menyediakan visualisasi waterfall dan dependence plot |
| Prompt Package | Menyiapkan input untuk LLM recommendation system |

Bagian prediction JSON berguna untuk analisis teknis karena berisi prediksi, aktual, error, context features, dan top SHAP contributors.

Bagian plot SHAP berguna untuk interpretasi visual, terutama ketika ingin menjelaskan mengapa suatu prediksi muncul pada waktu dan stasiun tertentu.

Bagian prompt package berguna untuk tahap aplikasi, karena prediksi numerik dan faktor penjelas sudah diubah menjadi input terstruktur untuk LLM.

Dengan demikian, notebook ini tidak hanya berhenti pada interpretasi model, tetapi juga mulai mengarah ke sistem rekomendasi operasional berbasis prediksi kualitas udara.

# Struktur Folder dan File Output Notebook

Notebook ini menghasilkan satu folder output utama:

```text
outputs_llm_xai/
```

Input utama notebook adalah:

```text
outputs_ablation_datasets/dataset_h6_ablation.csv
outputs_ablation_datasets/dataset_h12_ablation.csv
outputs_ablation_datasets/dataset_h24_ablation.csv
outputs_phase2/best_params_h6.json
outputs_phase2/best_params_h12.json
outputs_phase2/best_params_h24.json
airsafe_recommendation_system.txt
airsafe_recommendation_user.txt
```

Struktur output notebook dapat diringkas sebagai berikut:

```text
project/
│
├── outputs_ablation_datasets/
│   ├── dataset_h6_ablation.csv
│   ├── dataset_h12_ablation.csv
│   └── dataset_h24_ablation.csv
│
├── outputs_phase2/
│   ├── best_params_h6.json
│   ├── best_params_h12.json
│   └── best_params_h24.json
│
├── airsafe_recommendation_system.txt
├── airsafe_recommendation_user.txt
│
└── outputs_llm_xai/
    ├── prediction_json/
    │   ├── prediction_h6_with_top_shap.json
    │   ├── prediction_h6_with_top_shap.csv
    │   ├── prediction_h12_with_top_shap.json
    │   ├── prediction_h12_with_top_shap.csv
    │   ├── prediction_h24_with_top_shap.json
    │   ├── prediction_h24_with_top_shap.csv
    │   └── prediction_all_horizons_with_top_shap.json
    │
    ├── waterfall_plots/
    │   ├── waterfall_h6_*.png
    │   ├── waterfall_h12_*.png
    │   └── waterfall_h24_*.png
    │
    ├── dependence_plots/
    │   ├── dependence_h6_*.png
    │   ├── dependence_h12_*.png
    │   └── dependence_h24_*.png
    │
    ├── prompt_packages/
    │   ├── llm_prompt_input.csv
    │   ├── llm_prompt_packages.jsonl
    │   └── llm_prompt_packages.json
    │
    ├── waterfall_manifest.csv
    └── dependence_manifest.csv
```

## Penjelasan File Prediction JSON

| File | Fungsi |
|---|---|
| `prediction_h6_with_top_shap.json` | Prediksi H6 dengan top SHAP contributors |
| `prediction_h6_with_top_shap.csv` | Versi CSV ringkas prediksi H6 |
| `prediction_h12_with_top_shap.json` | Prediksi H12 dengan top SHAP contributors |
| `prediction_h12_with_top_shap.csv` | Versi CSV ringkas prediksi H12 |
| `prediction_h24_with_top_shap.json` | Prediksi H24 dengan top SHAP contributors |
| `prediction_h24_with_top_shap.csv` | Versi CSV ringkas prediksi H24 |
| `prediction_all_horizons_with_top_shap.json` | Gabungan prediksi H6, H12, dan H24 |

## Penjelasan File Plot SHAP

| Folder atau File | Fungsi |
|---|---|
| `waterfall_plots/` | Menyimpan waterfall plot SHAP per sample |
| `dependence_plots/` | Menyimpan dependence plot SHAP per stasiun dan fitur |
| `waterfall_manifest.csv` | Daftar file waterfall plot beserta horizon, stasiun, dan waktu |
| `dependence_manifest.csv` | Daftar file dependence plot beserta horizon, stasiun, dan fitur |

## Penjelasan File Prompt Package

| File | Fungsi |
|---|---|
| `llm_prompt_input.csv` | Tabel input gabungan H6, H12, H24 untuk prompt |
| `llm_prompt_packages.jsonl` | Prompt package format JSONL untuk batch inference |
| `llm_prompt_packages.json` | Prompt package format JSON list untuk inspeksi atau aplikasi |

Output paling penting dari notebook ini adalah:

```text
outputs_llm_xai/prediction_json/prediction_all_horizons_with_top_shap.json
outputs_llm_xai/prompt_packages/llm_prompt_input.csv
outputs_llm_xai/prompt_packages/llm_prompt_packages.jsonl
outputs_llm_xai/prompt_packages/llm_prompt_packages.json
```

File-file tersebut menjadi jembatan antara model forecasting, SHAP explainability, dan sistem rekomendasi berbasis LLM.